In [ ]:
from pathgen.utils.seeds import set_seed

global_seed = 13371
set_seed(global_seed)

In [ ]:
from pathgen.utils.paths import project_root

experiment_name = "paper"
experiment_root = project_root() / "experiments" / experiment_name

## 2. Sampling at a slide level
We want the samples in the different sets (train, validate, and test) to come from seperate slides. Test is it's own dataset, but validate has to be broken off from train on a per-slide basis. We want the split to take into account the number of patches in each set (because it is the patches that we are interested in). Let's base the split so that the validate set has about 30% of the tumor patches in it. Baseing it on the tumor patches makes sense because they are the class with the smallest number of samples.

Our algorithm for splitting will work like so:
1. Work out the total number of tumor patches in the whole dataset.
2. Work out what 30% (or whatever the amount is) of the total patches is.
3. Start counting with 0 patches and 0 slides in the validation set.
4. Randomly select a slide in training and move it to validation.
5. Add the number of patches in that slide to the total.
6. Repeat from 4 until the total number of patches is greater than 30% of the total.

In [ ]:
from pathgen.preprocess.patching import SlidesIndex

train_index = SlidesIndex.load(experiment_root / 'index' / 'train')
test_index = SlidesIndex.load(experiment_root / 'index' / 'test')
len(train_index), len(test_index)

In [ ]:
import random
from typing import Tuple

def split_on_slide(index: SlidesIndex, label: str, fraction: float) -> Tuple[SlidesIndex, SlidesIndex]:
    # compute the number of patches in each fraction
    summary = index.summary()
    total = summary[label].sum()
    split_count = int(total * fraction)
    
    # perform the split on the indices
    remaining_indices = list(range(len(summary)))
    random.shuffle(remaining_indices)
    split_indices = []
    total_split_patches = 0
    while total_split_patches < split_count:
        slide_idx = remaining_indices[0]  # front
        label_count = summary.iloc[slide_idx][label]
        total_split_patches += label_count
        remaining_indices.pop(0)
        split_indices.append(slide_idx)

    # create the indexes
    return index.select(remaining_indices), index.select(split_indices)

training_index, validation_index = split_on_slide(train_index, 'tumor', 0.3)
print(len(training_index))
print(len(validation_index))
training_index.summary()['tumor'].sum(), validation_index.summary()['tumor'].sum()

Now that we have the two sets, we can sample the patches from them.

## 3. Sampling patches from across all the slides

In [ ]:
# from pathgen.preprocess.patching import combine

from typing import List

import pandas as pd
import numpy as np

from pathgen.preprocess.patching import PatchSet

def combine(patchsets: List[PatchSet]) -> PatchSet:
    def to_frame(ps: PatchSet) -> pd.DataFrame:
        frame = ps.df.copy(deep=True)
        for attr in ["patch_size", "level", "slide_index", "dataset_name"]:
            value = getattr(ps, f"_{attr}", None)
            if attr not in frame.columns:
                frame[attr] = value
        return frame

    # create one big data frame with all the patch data in it
    frames = [to_frame(ps) for ps in patchsets]
    combined_df = pd.concat(frames, ignore_index=True)
    
    # optimise
    def is_unique(s):
        a = s.to_numpy()
        return (a[0] == a).all()

    cols = ["patch_size", "level", "slide_index", "dataset_name"]
    args = {}
    for col in cols:
        if is_unique(combined_df[col]):
            series = combined_df[col]
            args[col] = series[0]
            combined_df.pop(col)
    args["df"] = combined_df

    return PatchSet(**args)

training_patchset = combine(training_index.patches)
training_patchset.summary()

In [ ]:
validation_patchset = combine(validation_index.patches)
validation_patchset.summary()

In [ ]:
testing_patchset = combine(test_index.patches)
testing_patchset.summary()

In [ ]:
from pathgen.preprocess.sampling import sample

training_samples = sample(training_patchset, 35000)  # 70,000 total
training_samples.summary()

In [ ]:
validation_samples = sample(validation_patchset, 15000)  # 30,000 total
validation_samples.summary()

In [ ]:
testing_samples = sample(testing_patchset, 15000)  # 30,000 total
testing_samples.summary()

## 4. Export the patches to disk

In [ ]:
training_samples.export(experiment_root / 'patches' / 'train')

In [ ]:
print('validation')
validation_samples.export(experiment_root / 'patches' / 'valid')
print('testing')
testing_samples.export(experiment_root / 'patches' / 'test')

Let's display some of the patches to check they are ok

In [ ]:
import random
from PIL import Image

from IPython.display import Image
import pathlib

from pathlib import Path 
import matplotlib.pyplot as plt

def get_random_images(source_dir: Path, num_images: int) -> [Image]:
    image_paths = list(source_dir.glob('*.png'))
    print(source_dir)
    print(len(image_paths))
    return random.sample(image_paths, num_images)

def show_random_patches(source_dir: Path) -> None:
    images = get_random_images(source_dir, 9)
    
    plt.figure(figsize=(10,10))

    for i, img_path in enumerate(images):
      img_path = str(img_path)
    
      # the number of images in the grid is 3*3 (9)
      plt.subplot(3,3,i+1)
    
      img = plt.imread(img_path)
      plt.imshow(img)
      plt.title(Path(img_path).name)
      plt.axis("off")

    plt.show()
    plt.close()

In [ ]:
normal_train_patches_dir = experiment_root / 'patches' / 'train' / 'normal'
show_random_patches(normal_train_patches_dir)

In [ ]:
tumor_train_patches_dir = experiment_root / 'patches' / 'train' / 'tumor'
show_random_patches(tumor_train_patches_dir)